In [ ]:

dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")
dbutils.widgets.text("warehouse", "", "Warehouse")
dbutils.widgets.text("govern_table", "", "AI Governance output table")

In [ ]:
from uuid import uuid4

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse = dbutils.widgets.get("warehouse")
govern_table = dbutils.widgets.get("govern_table")

run_id = f"{uuid4()}"

In [ ]:
from governer.utils import AiResponse, task_genie
from databricks.sdk.service.dashboards import GenieSpace

def govern_table_comment(space: GenieSpace, table: str) -> AiResponse:
    table_comment_govern_prompt = f"""
        You are a data governance assistant for a Databricks workspace.

        Your task is to evaluate the existing comment for the table named {table}
        Use the table metadata, schema, column names, and any available sample data to determine whether the current comment is sufficient.
        Do not attempt to execute your recommendation or any data or metadata modifications.

        A table comment is considered SUFFICIENT only if it meets ALL of the following criteria:
        1. Clearly describes the business purpose of the table.
        2. Explains what type of data the table contains.
        3. Identifies the main entity or subject represented by the table.
        4. Provides context for how the table is typically used (analytics, reporting, ingestion stage, etc.).
        5. Is not empty, vague, or purely technical (e.g., "table for storing data", "generated table", etc.).

        If the comment meets all these criteria, respond with exactly:
        SUFFICIENT

        If the comment is missing, empty, or does not meet the criteria above, generate a concise but informative replacement comment based on the table schema and observed data.

        Your response must follow these strict rules:
        - Return ONLY a single SQL statement as plain text.
        - The SQL statement must add or update the table comment.
        - The SQL must be directly executable in a Databricks notebook.
        - Do not include explanations, markdown, formatting, or additional text.
        
        Example format:

        COMMENT ON TABLE {table} IS 'Clear business-oriented description of the table and its purpose.';
    """
    return task_genie(space, table_comment_govern_prompt)


In [ ]:
def evaluate_table_comment_governance_response(space: GenieSpace, table: str, response: AiResponse) -> float | None :
    evaluation_prompt = f"""
        You are a strict validation engine. Your ONLY task is to output a single numeric score.
        You are evaluating the correctness of another models output for Databricks table comment validation.

        Inputs:
        - Table name: {table}
        - Model output:
        "{response.text}"
        - Expected rules:
        "{response.prompt}"

        You may consider:
        - Table schema
        - Metadata
        - Existing comment
        - Table properties
        - Sample data (if available)

        You must NOT modify anything or execute any commands.

        Evaluation criteria:
        - Correct format (must strictly follow required SQL or "SUFFICIENT" rule)
        - Logical correctness vs schema/data
        - Proper use of "SUFFICIENT"
        - Quality and specificity of description (no vague/generic wording)

        Scoring:
        Return a probability between 0.0 and 1.0:

        1.0 = fully correct  
        0.8-0.99 = mostly correct, minor issues  
        0.5-0.79 = partially correct  
        0.2-0.49 = mostly incorrect  
        0.0-0.19 = completely incorrect  

        If evaluation is not possible, return:
        -1.0

        CRITICAL OUTPUT RULES (MUST FOLLOW):
        - Output ONLY a single number
        - No words, no explanations, no justification
        - No Markdown
        - No code blocks
        - No labels
        - No extra characters
        - No whitespace before or after
        - Do NOT restate the score
        - Do NOT provide suggestions
        - Do NOT format the output

        Valid examples:
        0.93
        0.4
        1.0
        -1.0

        INVALID examples:
        "0.93"
        Score: 0.93
        0.93 ✅
        ```0.93```
        0.93 (valid)

        FINAL INSTRUCTION:
        Return ONLY the number.
    """

    result = task_genie(space, evaluation_prompt)
    if not result.success:
        return None
    try:
        return float(result.text)
    except Exception:
        return None
     


In [ ]:
from governer.utils import get_valid_govern_tables

spark.sql(f"use catalog {catalog}")
spark.sql(f"use schema {schema}")

tables = get_valid_govern_tables(catalog=catalog, schema=schema, tables=spark.catalog.listTables())

In [ ]:
from governer.utils import GovernStatus, trash_genie_workspace, get_genie_workspace
from governer.schemas.governance.table_comment_actions import table_comment_actions
import datetime

res = []

genie_space_title="Genie Governer Space"
genie_space = get_genie_workspace(space_title=genie_space_title, warehouse_id=warehouse, tables=tables)

for table in tables:
    result = govern_table_comment(space=genie_space, table=table)
    print(f"{table}={result.text}")
    eval = evaluate_table_comment_governance_response(space=genie_space, table=table, response=result)
    res.append({
         "record_id": f"{uuid4()}",
         "run_id": run_id, 
         "table_name": table, 
         "govern_success": result.success, 
         "govern_error": result.error,
         "evaluation": eval,
         "sql": result.text, 
         "status": GovernStatus.GENERATED.name, 
         "timestamp": datetime.datetime.now()
     })

trash_genie_workspace(space=genie_space)

govern = spark.createDataFrame(data=res,schema=table_comment_actions)
govern.write.mode("append").saveAsTable(govern_table)